# Retail store inventory

link: https://www.kaggle.com/datasets/anirudhchauhan/retail-store-inventory-forecasting-dataset

In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv("../data/01_raw/retail_store_inventory.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df["item_id"] = df["Store ID"] + "_" + df["Product ID"]
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,item_id
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,S001_P0001
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,S001_P0002
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,S001_P0003
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,S001_P0004
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,S001_P0005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,2024-01-01,S005,P0016,Furniture,East,96,8,127,18.46,73.73,20,Snowy,0,72.45,Winter,S005_P0016
73096,2024-01-01,S005,P0017,Toys,North,313,51,101,48.43,82.57,10,Cloudy,0,83.78,Autumn,S005_P0017
73097,2024-01-01,S005,P0018,Clothing,West,278,36,151,39.65,11.11,10,Rainy,0,10.91,Winter,S005_P0018
73098,2024-01-01,S005,P0019,Toys,East,374,264,21,270.52,53.14,20,Rainy,0,55.80,Spring,S005_P0019


In [3]:
PREDICTION_LENGTH=15
TARGET="Units Sold"

In [4]:
train_data = TimeSeriesDataFrame.from_data_frame(
    df, id_column="item_id", timestamp_column="Date"
)

train_split, test_split = train_data.train_test_split(
    prediction_length=PREDICTION_LENGTH
)

predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=[0.025, 0.05, 0.5, 0.95 ,0.975],
    target=TARGET,
    eval_metric="WAPE",
)

predictor.fit(
    train_split,
    hyperparameters={
        "RecursiveTabular": {
            "tabular_hyperparameters": {"GBM": {}, "XGB": {}, "CAT": {}}
        },
        "Naive": {},
        "SeasonalNaive": {},
        "PatchTST": {},
        "ETS": {},
        "ADIDA": {},
        "DLinear": {},
        "Toto": {},
        "ARIMA": {},
        "Theta": {},
        "DeepAR": {},
        "NPTS": {},
        "TemporalFusionTransformer": {},
        "Chronos2": {},
        "SimpleFeedForward": {},
        "TiDE":{},
        "WaveNet": {},
    },
)

Sorting the dataframe index before generating the train/test split.
Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/walmart_sales_benchmark_mlops/notebooks/AutogluonModels/ag-20260518_041624'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.62/11.62 GB
Total GPU Memory:   Free: 11.62 GB, Allocated: 0.00 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       20.63 GB / 31.15 GB (66.2%)
Disk Space Avail:   111.02 GB / 230.30 GB (48.2%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                     'DLi

In [5]:
leaderboard = predictor.leaderboard(train_split)

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


In [6]:
leaderboard

,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-0.632043,-0.632417,22.445793,21.214891,0.269663,18
1,TemporalFusionTransformer,-0.634147,-0.634147,0.095158,0.054472,64.075390,8
2,Chronos2,-0.635701,-0.635701,4.802874,3.936686,2.487660,7
3,WaveNet,-0.635828,-0.636252,0.169269,0.146349,31.969716,12
4,Toto,-0.638252,-0.636193,20.966184,19.130924,1.687304,17
5,NPTS,-0.643565,-0.643565,1.077569,0.670609,0.017983,5
6,PatchTST,-0.643946,-0.643946,0.051166,0.038418,32.079010,10
7,TiDE,-0.644291,-0.644291,0.099550,0.071310,25.120116,11
8,SimpleFeedForward,-0.646477,-0.646477,0.046274,0.041688,13.695306,16
9,DLinear,-0.646868,-0.646868,0.050073,0.036043,18.135977,14


In [12]:
predictions = predictor.predict(train_split, model="WeightedEnsemble")

In [ ]:
selected_store = "S001"
selected_product = "P0002"
selected_item = f"{selected_store}_{selected_product}"

real_data = test_split.loc[selected_item]
predictions_store = predictions.loc[selected_item]

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=real_data.index,
        y=real_data[TARGET],  
        name="Actual Units Sold",
        mode="lines",
        line=dict(color="#1f77b4", width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["mean"],
        name="Forecast (Mean)",
        mode="lines",
        line=dict(color="#ff7f0e", width=2, dash="dash"),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.975"],
        name="Upper Bound (p97.5)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.025"],
        name="95% Confidence Interval (p2.5 - p97.5)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(255, 127, 14, 0.15)",
        line=dict(width=0),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.95"],
        name="Upper Bound (p95)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.05"],
        name="90% Confidence Interval (p5 - p95)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(200, 100, 14, 0.15)",
        line=dict(width=0),
    )
)

fig.update_layout(
    title=f"Backtesting (Actual vs Forecast) - {selected_item}",
    xaxis_title="Date",
    yaxis_title="Units Sold",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()